In [1]:
# 필수 라이브러리 설치
%pip install langchain chromadb langchain-google-genai pypdf PyPDF2 python-dotenv tqdm langchain-community --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 필요한 라이브러리 임포트
import os
import warnings
from dotenv import load_dotenv
from tqdm import tqdm

# PDF 처리를 위한 모듈 (원본 코드 기반)
from PyPDF2 import PdfReader
from langchain.document_loaders import PDFPlumberLoader

# LangChain 관련 모듈
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.docstore.document import Document

# 경고 메시지 숨기기 (선택 사항)
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
# --- 1. 환경 설정 ---
load_dotenv() # .env 파일 로드

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("오류: .env 파일 또는 환경 변수에서 GOOGLE_API_KEY를 찾을 수 없습니다.")
    exit()
else:
    print("Google API 키 로드 완료.")

# PDF 파일이 있는 폴더 경로 설정
pdf_folder_path = "house_pdf"  # 실제 PDF 파일들이 있는 폴더 경로로 수정하세요.
persist_directory = "chroma_db" # 벡터 DB를 저장할 디렉토리 이름

# PDF 폴더 존재 확인
if not os.path.isdir(pdf_folder_path):
    print(f"오류: 지정된 PDF 폴더 '{pdf_folder_path}'를 찾을 수 없습니다.")
    exit()

print(f"PDF 폴더 경로: {pdf_folder_path}")
print(f"벡터 DB 저장 경로: {persist_directory}")

Google API 키 로드 완료.
PDF 폴더 경로: house_pdf
벡터 DB 저장 경로: chroma_db


In [4]:
# --- 2. 문서 로드 및 처리 (원본 코드의 함수 사용) ---
def process_single_pdf(filepath, filename):
    """단일 PDF 파일을 처리하고 페이지 목록(Document 객체 리스트)을 반환합니다."""
    # print(f"단일 PDF 파일 처리 시작: {filename}") # tqdm 사용 시 개별 출력은 생략 가능
    pages_with_source = []
    try:
        # 파일 존재 확인
        if not os.path.exists(filepath):
            print(f"경고: 파일이 존재하지 않습니다: {filepath}")
            return []

        # PyPDF2로 메타데이터 추출
        pdf_info = {}
        total_pages = 0
        try:
            with open(filepath, "rb") as f:
                reader = PdfReader(f)
                if reader.is_encrypted:
                    try:
                         # 암호 해제 시도 (비밀번호가 없다면 실패)
                         if reader.decrypt('') == 0: # 0은 실패 의미
                             print(f"경고: 암호화된 PDF 파일(비밀번호 없음) 건너뜁니다: {filename}")
                             return []
                    except Exception as decrypt_error:
                         print(f"경고: 암호화된 PDF 파일 처리 불가 ({decrypt_error}): {filename}")
                         return []

                total_pages = len(reader.pages)

                if reader.metadata:
                    pdf_info['title'] = reader.metadata.get('/Title', '').strip()
                    pdf_info['author'] = reader.metadata.get('/Author', '').strip()
                    # 필요한 다른 메타데이터 필드 추가 가능
                pdf_info = {k: v for k, v in pdf_info.items() if v} # 빈 값 제거
        except Exception as e:
            print(f"경고: PyPDF2 메타데이터 읽기 오류: {filename} - {e}")
            # 메타데이터 로드 실패해도 내용 로드는 시도

        # PDF 내용 로드 (PDFPlumberLoader 사용)
        try:
            loader = PDFPlumberLoader(filepath)
            pages = loader.load() # 페이지별 Document 객체 리스트 반환

            if not pages:
                print(f"경고: {filename}: PDFPlumberLoader가 페이지를 추출하지 못했습니다.")
                return []

            # 메타데이터 병합 및 Document 객체 생성
            for i, page in enumerate(pages):
                merged_metadata = {
                    'source': filepath, # PyPDFLoader와 달리 PDFMiner는 기본 source가 없을 수 있음
                    'filename': filename, # 파일명 추가
                    'page': i + 1, # 페이지 번호 (0부터 시작하므로 +1)
                    'total_pages': total_pages
                }
                merged_metadata.update(pdf_info) # PyPDF2로 추출한 정보 추가

                # 원본 페이지의 메타데이터도 존중 (예: PDFMiner가 자체적으로 추출한 정보)
                if page.metadata:
                    # source, page 키는 우리가 설정한 것으로 덮어쓰고 나머지는 추가
                    original_meta_filtered = {k: v for k, v in page.metadata.items() if k not in ['source', 'page']}
                    merged_metadata.update(original_meta_filtered)

                pages_with_source.append(
                    Document(
                        page_content=page.page_content,
                        metadata=merged_metadata
                    )
                )
            # print(f"  - {filename}: {len(pages_with_source)} 페이지 Document 생성 (총 {total_pages} 페이지)")
        except Exception as e:
            print(f"경고: PDFMinerLoader 로딩 오류: {filename} - {e}")
            return []

    except Exception as e:
        print(f"경고: PDF 처리 중 예외 발생: {filename} - {e}")
        return []

    return pages_with_source

In [5]:
# PDF 폴더 내 모든 PDF 파일 처리
def process_pdfs(path):
    """지정된 경로(폴더)에서 PDF 파일을 로드하고 처리합니다."""
    all_pages = []

    if not os.path.isdir(path):
        print(f"경고: 폴더 경로 '{path}'가 유효하지 않습니다.")
        return []

    pdf_files = [f for f in os.listdir(path) if f.lower().endswith('.pdf')]
    if not pdf_files:
        print(f"경고: '{path}' 폴더에 PDF 파일이 없습니다.")
        return []

    print(f"처리할 PDF 파일 {len(pdf_files)}개 발견.")

    for pdf_filename in tqdm(pdf_files, desc="PDF 처리 중"):
        filepath = os.path.join(path, pdf_filename)
        pages = process_single_pdf(filepath, pdf_filename)
        all_pages.extend(pages)

    print(f"총 {len(all_pages)} 페이지 (Document 객체) 처리 완료.")
    return all_pages

# PDF 처리 실행
global_pages = process_pdfs(pdf_folder_path)

if global_pages:
    print("처리된 첫 페이지 메타데이터 예시:", global_pages[0].metadata)

처리할 PDF 파일 16개 발견.


PDF 처리 중: 100%|██████████| 16/16 [01:49<00:00,  6.83s/it]

총 415 페이지 (Document 객체) 처리 완료.
처리된 첫 페이지 메타데이터 예시: {'source': 'house_pdf\\(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'filename': '(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'page': 1, 'total_pages': 19, 'file_path': 'house_pdf\\(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'Creator': 'Hwp 2020 11.0.0.7571', 'Producer': 'Hancom PDF 1.3.0.546', 'CreationDate': "D:20240520135921+09'00'", 'ModDate': "D:20240520135921+09'00'", 'PDFVersion': '1.4'}


In [6]:
# --- 3. 문서 분할 ---
if not global_pages:
     print("오류: 분할할 문서가 없습니다. PDF 처리 단계를 확인하세요.")
     exit()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)
split_docs = text_splitter.split_documents(global_pages)

print(f"총 {len(global_pages)} 페이지를 {len(split_docs)}개의 청크로 분할 완료.")
if split_docs:
    print("분할된 첫 청크 메타데이터 예시:", split_docs[0].metadata)

총 415 페이지를 1741개의 청크로 분할 완료.
분할된 첫 청크 메타데이터 예시: {'source': 'house_pdf\\(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'filename': '(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'page': 1, 'total_pages': 19, 'file_path': 'house_pdf\\(영등포복합청사)행복주택 입주자 모집 공고문(2024.5.31.).pdf', 'Creator': 'Hwp 2020 11.0.0.7571', 'Producer': 'Hancom PDF 1.3.0.546', 'CreationDate': "D:20240520135921+09'00'", 'ModDate': "D:20240520135921+09'00'", 'PDFVersion': '1.4'}


In [7]:
# --- 4. 임베딩 및 벡터 저장 ---
try:
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=google_api_key)
    print("Gemini 임베딩 모델 초기화 완료.")
except Exception as e:
    print(f"Gemini 임베딩 모델 초기화 오류: {e}")
    exit()

if os.path.exists(persist_directory):
    print(f"기존 벡터 저장소 '{persist_directory}'를 로드합니다.")
    vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
    print(f"기존 벡터 저장소 로드 완료. 저장된 문서 수: {vectordb._collection.count()}")
else:
    if not split_docs:
        print("오류: 벡터 저장소를 생성할 분할된 문서가 없습니다.")
        exit()
    print(f"'{persist_directory}'에 새로운 벡터 저장소를 생성하고 문서를 저장합니다.")
    vectordb = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    print(f"새로운 벡터 저장소 생성 및 저장 완료. 저장된 문서 수: {vectordb._collection.count()}")

Gemini 임베딩 모델 초기화 완료.
'chroma_db'에 새로운 벡터 저장소를 생성하고 문서를 저장합니다.
새로운 벡터 저장소 생성 및 저장 완료. 저장된 문서 수: 1741


In [8]:
# --- 5. LLM 및 Retriever 초기화 ---
try:
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        google_api_key=google_api_key,
        temperature=0.1, # Few-shot 예제를 따르도록 낮은 온도 설정 권장
        convert_system_message_to_human=True
    )
    print("Gemini LLM 초기화 완료.")
except Exception as e:
    print(f"Gemini LLM 초기화 오류: {e}")
    exit()

retriever = vectordb.as_retriever(
    search_type="mmr", # MMR 사용
    search_kwargs={'k': 10, 'fetch_k': 500} # 최종 5개를 선택하기 위해 20개를 먼저 가져옴
)
print(f"Retriever 초기화 완료")

Gemini LLM 초기화 완료.
Retriever 초기화 완료


In [9]:
# --- 6. RAG 체인 생성 ---
prompt_template_str = """
당신은 한국의 부동산 정책과 입주 공고 전문가입니다. 제공된 "문맥 정보"를 바탕으로 "질문"에 대해 답변해주세요.

지침:

- 답변은 무조건 한국어로만 답하세요.
-너무 길거나 복잡한 답변은 피하세요. 사용자가 이해하기 쉽도록 작성하세요.
-"문맥 정보"에서 질문에 대한 답을 찾으세요.
-답을 찾을 수 없는 경우, "죄송합니다, 제공된 문서에서 해당 질문에 대한 답변을 찾을 수 없습니다."라고 명확히 답변하세요.
-절대 추측하거나 주어진 정보 외의 내용을 답변하지 마세요.
-답변 생성 시 반드시 제공된 "문맥 정보"만을 근거로 하세요.
-답변은 친절하고 명확하게 한국어로 작성해주세요.
-가능하다면, 답변의 근거가 된 문서의 출처(파일명, 페이지 번호)를 답변 끝에 간략히 언급하세요. (예: [출처: 파일명.pdf, 페이지번호])

문맥 정보:
{context}

질문:
{question}

답변:
"""

# PromptTemplate 객체 생성
QA_PROMPT = PromptTemplate(
    template=prompt_template_str, input_variables=["context", "question"]
)

# RetrievalQA 체인 생성 (단순화된 프롬프트 사용)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine", # Stuff 대신 Refine 사용 (Map_Reduce도 가능)
    retriever=retriever,
    return_source_documents=True
)
print(f"Refine 체인 (k=10) 생성 완료.")


Refine 체인 (k=10) 생성 완료.


In [10]:
# --- 7. 질의응답 실행 ---
print("주택 공고에 대해 질문해주세요. 종료하려면 '종료'를 입력하세요.")

while True:
    try:
        user_question = input("\n👤 질문: ")
        if user_question.lower() == "종료":
            print("\n🤖 챗봇: 이용해주셔서 감사합니다. 종료합니다.")
            break
        if not user_question.strip():
            print("질문을 입력해주세요.")
            continue

        print("\n⏳ 답변 생성 중...")

        result = qa_chain({"query": user_question})

        print("\n🤖 답변:")
        print(result["result"])

        print("\n📚 근거 문서:")
        if result.get("source_documents"):
            unique_sources = set()
            for doc in result["source_documents"]:
                source_info = f"- 파일: {doc.metadata.get('filename', '알 수 없음')}, 페이지: {doc.metadata.get('page', '알 수 없음')}"
                if source_info not in unique_sources:
                    print(source_info)
                    unique_sources.add(source_info)
        else:
            print("근거 문서를 찾을 수 없습니다.")

    except Exception as e:
        print(f"\n⚠️ 처리 중 오류 발생: {e}")
    except KeyboardInterrupt:
        print("\n\n🤖 챗봇: 사용자에 의해 중단되었습니다. 종료합니다.")
        break

주택 공고에 대해 질문해주세요. 종료하려면 '종료'를 입력하세요.

⏳ 답변 생성 중...


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_1996\1247305930.py:16: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": user_question})



🤖 답변:
The new context provides information about specific unit types (36A and 36B) within a 행복주택 complex, specifically designated for "주거약자" (vulnerable individuals) and 신혼부부/한부모가족 (newlyweds/single-parent families). It also details the lease terms (deposit and rent).

This context **does not change** the fundamental ineligibility of a 34-year-old married man without children for the 청년 or 사회초년생 categories. The new information focuses on *different* eligibility categories (주거약자, 신혼부부/한부모가족) that he also doesn't appear to meet based on the original question.

Therefore, the original answer remains the most accurate and helpful response to the user's question. The new context, while informative about specific unit types and lease terms, doesn't open up any new avenues for eligibility based on the information provided about the user.

**Original Answer:**

제공된 정보에 따르면, 34세 기혼 남성은 수서 지역 행복주택 중 청년 또는 사회초년생 자격으로는 신청이 **불가능**합니다. 그 이유는 다음과 같습니다.

*   **청년 자격:** 19세 이상 39세 이하의 미혼자만 해당됩니다. (➀-